In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
import datetime
from src.get_rda_era5 import ERA5DataSource
import re
import zarr

In [2]:
# --- risk days
pph = xr.load_dataset("data/raw_data/labelled_pph.nc")
missing_dates = [
    '200204250000', '200208300000', '200304150000', '200304160000',
    '200306250000', '200307270000', '200307280000', '200312280000',
    '200404140000', '200408090000', '200905280000', '201105210000',
    '202005240000', '200510240000'
]
dates_of_interest = pph["time"][pph["MAX_CAT"].isin(['SLGT', 'ENH', 'MDT', 'HIGH'])]
dates_of_interest = dates_of_interest[dates_of_interest > "200203310000"]
dates_of_interest = dates_of_interest[~(dates_of_interest.isin(missing_dates))]
selected_days = pd.to_datetime(dates_of_interest.values, format="%Y%m%d%H%M").normalize()

years = np.unique(selected_days.year)

LONG_TO_SHORT = {
    # PRESSURE LEVEL VARS
    "geopotential": "z",
    "specific_humidity": "q",
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
    "vertical_velocity": "w"
}

SHORT_TO_LONG = {value: key for key, value in LONG_TO_SHORT.items()}

def longname_to_channel(longname, level=None):
    """
    Return a channel string e.g. "u10m", "t850", "z500".
    If longname corresponds to a surface variable (10m, 2m, etc) no level needed.
    If it is a pressure-variable and level given -> return e.g. "t850".
    """
    # surface pattern
    if longname in LONG_TO_SHORT:
        short = LONG_TO_SHORT[longname]
        if level is None:
            return short
        else:
            return f"{short}{int(level)}"
    raise KeyError(f"No conversion known for {longname}; add to LONG_TO_SHORT")


def channels_for_config():
    return (
        # pressure-level variables (expanded over levels)
        [longname_to_channel(v, lvl)
         for v in ["geopotential", "specific_humidity", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"]
         if LONG_TO_SHORT.get(v, "")
         for lvl in [850, 500]]
    )

In [3]:
all_channels = channels_for_config()

In [4]:
def channel_da_to_dataset(da):
    """
    Convert channel-stacked DataArray into final Dataset.

    """

    ds_out = {}

    for ch in da.channel.values:
        sub = da.sel(channel=ch).drop_vars("channel")

        if bool(re.search(r'\d{3}$', ch)):
            level = ch[-3:]
            longname = SHORT_TO_LONG[ch[:-3]]
        else:
            level = None
            longname = SHORT_TO_LONG[ch]

        if level is None:
            # surface variable
            ds_out.setdefault(longname, []).append(sub)
        else:
            # pressure-level variable
            sub = sub.assign_coords(level=level).expand_dims("level")
            ds_out.setdefault(longname, []).append(sub)

    data_vars = {}

    for name, pieces in ds_out.items():
        merged = xr.concat(pieces, dim="level") if "level" in pieces[0].dims else pieces[0]
        data_vars[name] = merged

    return xr.Dataset(data_vars)

In [5]:
out_dir = Path("data/raw_data/clustering_glade.zarr")

# Skip everything except the most recent day in the store (always redo the last day). Can pick up where it left off
done = set()
if out_dir.exists():
    d = np.asarray(xr.open_zarr(out_dir, consolidated=False)["day"].values).astype("datetime64[ns]")
    if d.size:
        done = set(d[d < d.max()])

    
ds = ERA5DataSource(all_channels)
store_exists = out_dir.exists()

for day in selected_days:
    d64 = np.datetime64(day.to_datetime64()).astype("datetime64[ns]")
    if d64 in done:
        continue

    print(f"Processing {day.date()}")
    t = pd.Timestamp(day).normalize() + pd.Timedelta(days=1)

    day_ds = (
        ds[t]
        .isel(time=0, drop=True)
        .sel(
            latitude=slice(60.5, 13.25),
            longitude=slice(360 - 132.5, 360 - 56.75),
        )
        .expand_dims(day=[pd.Timestamp(day).normalize().to_datetime64()])
        .transpose("day", "channel", "latitude", "longitude")
    )

    day_ds.to_zarr(
        out_dir,
        mode="a" if store_exists else "w",
        append_dim="day" if store_exists else None,
        consolidated=False,
    )
    store_exists = True  # after first successful write

zarr.consolidate_metadata(out_dir)

Processing 2020-10-23
Processing 2020-10-28
Processing 2020-11-10
Processing 2020-11-14


Process ForkProcess-27:
Process ForkProcess-28:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker


KeyboardInterrupt: 

In [ ]:
# To fix if needed

store = zarr.open("data/raw_data/clustering_glade.zarr", mode="r+")

# True length = length of the coordinate
true_len = store["day"].shape[0]

for name, arr in store.arrays():
    if "day" in arr.attrs.get("_ARRAY_DIMENSIONS", []):
        if arr.shape[0] > true_len:
            print(f"Trimming {name}: {arr.shape[0]} -> {true_len}")
            arr.resize((true_len, *arr.shape[1:]))
zarr.consolidate_metadata("data/raw_data/clustering_glade.zarr")
